# 45 · Map's lines between records, checked against pandas

## What this checks

**Map** can draw a line from one record to another. Here each case names its infector in the `infector` column, and
Map finds that record by its `case_ID`. It draws a line from the infector's house to the case's house. This check
covers release 0.49.0.

Map also counts every record under one outcome. A line may be drawn. Both ends may be at one place. The record may
name no one, name a case the table does not hold, or name itself. It may name an id held twice. One end may have no
location.

This check covers those lines and counts, not the rest of the map. A missing or extra line shows a spread that did
not happen.

## How it is checked

We rebuild the lines with **pandas**, Python's library for tables. pandas joins the table of cases to itself, each
infector to the matching `case_ID`. Then it compares each pair's two locations. No code is shared with Fieldwork.

The data are the measles outbreak in Hagelloch, Germany, in 1861: 188 records. The houses are placed by this
project's own reading of the village map.

## What should happen

1. **The same lines.** The same set of infector and case pairs is drawn, and no other.
2. **The same counts** for every outcome listed above.
3. **Every record counted once.** The outcomes add up to the number of records.

These criteria were fixed before the check was run.

In [ ]:
NEEDED = ["pandas"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the installed packages.")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_45_map_links import run

report = run(Path.cwd() / "fixtures" / "links-hagelloch-001.json")
print("agrees:", report["agrees"])
print(report["records"], "records")
for outcome, n in report["counts"].items():
    print(f"  {outcome:13} {n}")
print("lines only here:", report["linesOnlyHere"])
print("lines only in Fieldwork:", report["linesOnlyInFieldwork"])

## The result on record

First recorded on 10 November 2026, against release 0.23.1. Rerun on 26 November 2026, against release 0.49.0:
**Agrees**.

Of the 188 records, 94 lines were drawn. In 90 more, both ends are in one house: two children of the same house.
4 records name no infector. The 94 lines are the same as pandas finds, and there are no others.

The result agrees in CPython and in the browser kernel.

Running the cell above repeats the check on the same data.

## What this does not show

- Who infected whom. Both sides read the same 1861 reconstruction in the infector column.
- Where each house stood. Both sides read this project's placing of the houses.
- That the lines are drawn as computed. This checks the work the figure is drawn from, not the figure.
- The outcomes Hagelloch does not have. No case names an unknown case, itself, an id held twice, or a house with no
  place. So those four counts agree only at zero. Fieldwork's own tests cover them.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
e = report["environment"]
print("pandas  :", e["pandas"])
print("runtime :", e["runtime"], "· Pyodide" if e["pyodide"] else "")